# Vehicle Detection and Tracking System

## Author: Based on Candy Calorie Counter by Evan Juras, EJ Technology Consultants

## Description:
This script uses YOLO vehicle detection model to locate and identify vehicles in recorded video files.
It uses a homemade StrongSORT tracker to track vehicles across frames and counts the total number
of vehicles detected on highway footage. The system processes video files and outputs annotated
video with vehicle counts and tracking information. 
## Requirements:
- Python 3.x
- OpenCV
- NumPy
- YOLOv5 or YOLOv8 model files
- Homemade StrongSORT tracker implementation

## evironment usage
-OfeDeM(python 3.11.9)

In [3]:
%pip install ultralytics

Note: you may need to restart the kernel to use updated packages.


In [1]:
# Import necessary packages
import os
import sys
import cv2
import numpy as np
from ultralytics import YOLO
from collections import defaultdict, deque
import math
import torch
import torchvision.transforms.functional as F
from torchvision.models.optical_flow import raft_large, Raft_Large_Weights
import logging

# Configure logging for debugging
logging.basicConfig(level=logging.INFO)
print(f"numpy version: {np.__version__}")
print("All required libraries imported successfully!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name()}")

numpy version: 2.2.6
All required libraries imported successfully!
PyTorch version: 2.5.1+cu121
CUDA available: True
CUDA device: NVIDIA GeForce GTX 1070


In [2]:
# Homemade StrongSORT Tracker Implementation
class HomemadeStrongSORT:
    def __init__(self, max_disappeared=30, max_distance=100):
        """
        Initialize the StrongSORT tracker
        
        Args:
            max_disappeared (int): Maximum number of frames a track can be lost before deletion
            max_distance (float): Maximum distance for associating detections with existing tracks
        """
        self.next_object_id = 0
        self.objects = {}  # Dictionary to store tracked objects
        self.disappeared = {}  # Track how many frames each object has been missing
        self.max_disappeared = max_disappeared
        self.max_distance = max_distance
        
    def register(self, centroid, bbox, class_id, confidence):
        """Register a new object with given centroid, bounding box, and confidence"""
        self.objects[self.next_object_id] = {
            'centroid': centroid,
            'bbox': bbox,
            'class_id': class_id,
            'confidence': confidence,
            'trail': deque(maxlen=10)  # Store last 10 positions for trail
        }
        self.objects[self.next_object_id]['trail'].append(centroid)
        self.disappeared[self.next_object_id] = 0
        self.next_object_id += 1
        
    def deregister(self, object_id):
        """Remove an object from tracking"""
        del self.objects[object_id]
        del self.disappeared[object_id]
        
    def update(self, detections):
        """
        Update the tracker with new detections
        
        Args:
            detections (list): List of detections, each containing [x1, y1, x2, y2, conf, class_id]
        
        Returns:
            dict: Updated objects with tracking IDs
        """
        # If no detections, mark all existing objects as disappeared
        if len(detections) == 0:
            for object_id in list(self.disappeared.keys()):
                self.disappeared[object_id] += 1
                if self.disappeared[object_id] > self.max_disappeared:
                    self.deregister(object_id)
            return self.objects
        
        # Calculate centroids for all detections
        input_centroids = []
        input_bboxes = []
        input_classes = []
        input_confidences = []
        
        for detection in detections:
            x1, y1, x2, y2, conf, class_id = detection
            cx = int((x1 + x2) / 2.0)
            cy = int((y1 + y2) / 2.0)
            input_centroids.append((cx, cy))
            input_bboxes.append((x1, y1, x2, y2))
            input_classes.append(class_id)
            input_confidences.append(conf)
        
        # If no existing objects, register all detections as new objects
        if len(self.objects) == 0:
            for i in range(len(input_centroids)):
                self.register(input_centroids[i], input_bboxes[i], input_classes[i], input_confidences[i])
        else:
            # Compute distance matrix between existing objects and new detections
            object_centroids = [obj['centroid'] for obj in self.objects.values()]
            object_ids = list(self.objects.keys())
            
            # Calculate Euclidean distances
            D = np.linalg.norm(np.array(object_centroids)[:, np.newaxis] - np.array(input_centroids), axis=2)
            
            # Find the minimum distance associations
            rows = D.min(axis=1).argsort()
            cols = D.argmin(axis=1)[rows]
            
            # Keep track of used row and column indices
            used_row_indices = set()
            used_col_indices = set()
            
            # Update existing objects with closest detections
            for (row, col) in zip(rows, cols):
                if row in used_row_indices or col in used_col_indices:
                    continue
                
                if D[row, col] > self.max_distance:
                    continue
                
                # Update the object
                object_id = object_ids[row]
                self.objects[object_id]['centroid'] = input_centroids[col]
                self.objects[object_id]['bbox'] = input_bboxes[col]
                self.objects[object_id]['class_id'] = input_classes[col]
                self.objects[object_id]['confidence'] = input_confidences[col]  # Update confidence
                self.objects[object_id]['trail'].append(input_centroids[col])
                self.disappeared[object_id] = 0
                
                used_row_indices.add(row)
                used_col_indices.add(col)
            
            # Handle unmatched detections and objects
            unused_row_indices = set(range(0, D.shape[0])).difference(used_row_indices)
            unused_col_indices = set(range(0, D.shape[1])).difference(used_col_indices)
            
            # If there are more objects than detections, mark objects as disappeared
            if D.shape[0] >= D.shape[1]:
                for row in unused_row_indices:
                    object_id = object_ids[row]
                    self.disappeared[object_id] += 1
                    
                    if self.disappeared[object_id] > self.max_disappeared:
                        self.deregister(object_id)
            
            # If there are more detections than objects, register new objects
            else:
                for col in unused_col_indices:
                    self.register(input_centroids[col], input_bboxes[col], input_classes[col], input_confidences[col])
        
        return self.objects

print("HomemadeStrongSORT tracker class defined successfully!")

HomemadeStrongSORT tracker class defined successfully!


In [7]:
# Configuration Parameters
# Define path to model and other user variables
model_path = r'C:\Users\Taber\Desktop\Nutt_KMUTT_folder\Vehicle_tracking\yolo11s.pt'  # Path to YOLO model (you can replace with your custom model)
input_video_path = ''  # Will be set dynamically - input video file path
output_video_path = 'vehicle_tracking_output.mp4'  # Output video file path
min_thresh = 0.30  # Minimum detection threshold
record_fps = 120  # FPS for output video

# Vehicle classes that we want to detect on highway (COCO dataset class IDs)
# You can modify this based on your custom model classes
vehicle_classes = {
    2: 'car',
    3: 'motorcycle', 
    5: 'bus',
    7: 'truck',
    1: 'bicycle'  # Sometimes bicycles are on highways too
}

# Class names for display (modify based on your model)
vehicle_display_names = {
    2: 'Car',
    3: 'Motorcycle',
    5: 'Bus', 
    7: 'Truck',
    1: 'Bicycle'
}

print("Configuration parameters set!")
print(f"Vehicle classes to detect: {list(vehicle_display_names.values())}")

Configuration parameters set!
Vehicle classes to detect: ['Car', 'Motorcycle', 'Bus', 'Truck', 'Bicycle']


In [8]:
# Helper Functions
def draw_trail(frame, trail_points, color, thickness=2):
    """Draw tracking trail for an object"""
    if len(trail_points) > 1:
        pts = np.array(list(trail_points), dtype=np.int32)
        pts = pts.reshape((-1, 1, 2))
        cv2.polylines(frame, [pts], False, color, thickness)

def draw_info_panel(frame, vehicle_counts, total_vehicles, fps=0):
    """Draw information panel with vehicle counts"""
    panel_height = 150
    panel_width = 400
    
    # Draw semi-transparent background
    overlay = frame.copy()
    cv2.rectangle(overlay, (10, 10), (panel_width, panel_height), (0, 0, 0), -1)
    cv2.addWeighted(overlay, 0.7, frame, 0.3, 0, frame)
    
    # Draw border
    cv2.rectangle(frame, (10, 10), (panel_width, panel_height), (255, 255, 255), 2)
    
    y_offset = 35
    # Total vehicles
    cv2.putText(frame, f'Total Vehicles Tracked: {total_vehicles}', 
                (20, y_offset), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
    y_offset += 25
    
    # Individual vehicle counts
    for class_id, count in vehicle_counts.items():
        if class_id in vehicle_display_names and count > 0:
            vehicle_name = vehicle_display_names[class_id]
            cv2.putText(frame, f'{vehicle_name}: {count}', 
                        (20, y_offset), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
            y_offset += 20
    
    # FPS
    if fps > 0:
        cv2.putText(frame, f'FPS: {fps:.1f}', 
                    (20, panel_height - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)

def get_bbox_colors():
    """Get predefined colors for bounding boxes"""
    return [
        (255, 0, 0),     # Red
        (0, 255, 0),     # Green  
        (0, 0, 255),     # Blue
        (255, 255, 0),   # Yellow
        (255, 0, 255),   # Magenta
        (0, 255, 255),   # Cyan
        (128, 0, 128),   # Purple
        (255, 165, 0),   # Orange
        (255, 192, 203), # Pink
        (0, 128, 0)      # Dark Green
    ]

print("Helper functions defined successfully!")

Helper functions defined successfully!


In [9]:
# Main Vehicle Detection and Tracking Function
def process_video_with_tracking(input_path, output_path, model_path, min_confidence=0.3):
    """
    Process video with vehicle detection and tracking
    
    Args:
        input_path (str): Path to input video file
        output_path (str): Path to output video file  
        model_path (str): Path to YOLO model
        min_confidence (float): Minimum confidence threshold for detections
    """
    
    # Check if model file exists
    if not os.path.exists(model_path):
        print(f'ERROR: Model path {model_path} is invalid or model was not found.')
        return False
        
    # Check if input video exists
    if not os.path.exists(input_path):
        print(f'ERROR: Input video {input_path} was not found.')
        return False
    
    print(f"Loading YOLO model from: {model_path}")
    # Load the YOLO model
    model = YOLO(model_path, task='detect')
    
    print(f"Opening input video: {input_path}")
    # Initialize video capture
    cap = cv2.VideoCapture(input_path)
    
    if not cap.isOpened():
        print("ERROR: Could not open input video file.")
        return False
    
    # Get video properties
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    print(f"Video properties: {width}x{height} @ {fps} FPS, Total frames: {total_frames}")
    
    # Initialize video writer
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))
    
    # Initialize tracker
    tracker = HomemadeStrongSORT(max_disappeared=30, max_distance=150)
    
    # Initialize counters
    frame_count = 0
    total_vehicles_ever_seen = set()  # Track unique vehicle IDs
    vehicle_counts = defaultdict(int)  # Count by vehicle type
    bbox_colors = get_bbox_colors()
    
    print("Starting video processing...")
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break
            
        frame_count += 1
        
        # Print progress every 100 frames
        if frame_count % 100 == 0:
            progress = (frame_count / total_frames) * 100
            print(f"Processing frame {frame_count}/{total_frames} ({progress:.1f}%)")
        
        # Run YOLO detection
        results = model(frame, verbose=False)
        detections = results[0].boxes
        
        # Process detections - filter for vehicle classes only
        current_detections = []
        
        if detections is not None and len(detections) > 0:
            for i in range(len(detections)):
                # Get bounding box coordinates
                xyxy_tensor = detections[i].xyxy.cpu()
                xyxy = xyxy_tensor.numpy().squeeze()
                x1, y1, x2, y2 = xyxy.astype(int)
                
                # Get class and confidence
                class_id = int(detections[i].cls.item())
                confidence = detections[i].conf.item()
                
                # Only process vehicle classes with sufficient confidence
                if class_id in vehicle_classes and confidence > min_confidence:
                    current_detections.append([x1, y1, x2, y2, confidence, class_id])
        
        # Update tracker with current detections
        tracked_objects = tracker.update(current_detections)
        
        # Draw tracking results
        current_frame_counts = defaultdict(int)
        
        for object_id, obj_data in tracked_objects.items():
            total_vehicles_ever_seen.add(object_id)
            
            x1, y1, x2, y2 = obj_data['bbox']
            class_id = obj_data['class_id']
            centroid = obj_data['centroid']
            trail = obj_data['trail']
            
            # Count vehicles in current frame
            current_frame_counts[class_id] += 1
            
            # Get color for this object
            color = bbox_colors[object_id % len(bbox_colors)]
            
            # Draw bounding box
            cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), color, 2)
            
            # Draw tracking trail
            if len(trail) > 1:
                draw_trail(frame, trail, color, thickness=2)
            
            # Draw object ID and class label
            if class_id in vehicle_display_names:
                label = f'ID:{object_id} {vehicle_display_names[class_id]} {int(confidence * 100)}%'
                label_size, _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
                
                # Draw label background
                cv2.rectangle(frame, (int(x1), int(y1) - label_size[1] - 10), 
                             (int(x1) + label_size[0], int(y1)), color, -1)
                
                # Draw label text
                cv2.putText(frame, label, (int(x1), int(y1) - 5), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
                
                # Draw center point
                cv2.circle(frame, centroid, 4, color, -1)
        
        # Update overall vehicle counts (keep track of maximum seen for each type)
        for class_id, count in current_frame_counts.items():
            vehicle_counts[class_id] = max(vehicle_counts[class_id], count)
        
        # Draw information panel(optional)
        #draw_info_panel(frame, vehicle_counts, len(total_vehicles_ever_seen), 
                       #fps if frame_count > 0 else 0)
        
        # Write frame to output video
        out.write(frame)
    
    # Clean up
    cap.release()
    out.release()
    
    print(f"\\nVideo processing completed!")
    print(f"Output saved to: {output_path}")
    print(f"Total unique vehicles tracked: {len(total_vehicles_ever_seen)}")
    print("Vehicle type breakdown:")
    for class_id, count in vehicle_counts.items():
        if class_id in vehicle_display_names:
            print(f"  {vehicle_display_names[class_id]}: {count}")
    
    return True

print("Main processing function defined successfully!")

Main processing function defined successfully!


In [10]:
# Fix numpy availability issue

# Import required modules to ensure they're available
import numpy as np
import cv2
import os

# Example Usage - Process a Video File
# Set your input video path here
input_video = r"C:\Users\Taber\Desktop\Nutt_KMUTT_folder\7\KEN-5895\FILE240403-085108F_lane_change_found.MP4"  # Change this to your video file path
output_video = r"C:\Users\Taber\Desktop\Nutt_KMUTT_folder\7\KEN-5895\FILE240403-085108F_Demo_with_trackingg.MP4"  # Output will be saved with this name

# Check if the input video file exists
if os.path.exists(input_video):
    print(f"Found input video: {input_video}")
    
    # Process the video
    success = process_video_with_tracking(
        input_path=input_video,
        output_path=output_video, 
        model_path=model_path,
        min_confidence=min_thresh
    )
    
    if success:
        print("\\n✅ Video processing completed successfully!")
        print(f"📁 Output video saved as: {output_video}")
        print("\\n🚗 The output video includes:")
        print("   • Vehicle detection and tracking")
        print("   • Unique vehicle ID assignment") 
        print("   • Vehicle counting by type")
        print("   • Tracking trails")
        print("   • Real-time statistics overlay")
    else:
        print("❌ Video processing failed!")
        
else:
    print(f"❌ Input video not found: {input_video}")
    print("\\n📋 Available video files in workspace:")
    
    # List available video files
    video_extensions = ['.mp4', '.avi', '.mkv', '.mov', '.MOV', '.MP4']
    video_files = []
    
    for root, dirs, files in os.walk(r"c:\Users\Taber\Desktop\Nutt_KMUTT_folder"):
        for file in files:
            if any(file.lower().endswith(ext.lower()) for ext in video_extensions):
                video_files.append(os.path.join(root, file))
    
    for i, video_file in enumerate(video_files[:10]):  # Show first 10 videos
        print(f"   {i+1}. {video_file}")
    
    if len(video_files) > 10:
        print(f"   ... and {len(video_files) - 10} more videos")
    
    print("\\n💡 To use this code:")
    print("   1. Copy one of the video file paths above") 
    print("   2. Update the 'input_video' variable in this cell")
    print("   3. Run this cell again")

Found input video: C:\Users\Taber\Desktop\Nutt_KMUTT_folder\7\KEN-5895\FILE240403-085108F_lane_change_found.MP4
Loading YOLO model from: C:\Users\Taber\Desktop\Nutt_KMUTT_folder\Vehicle_tracking\yolo11s.pt
Opening input video: C:\Users\Taber\Desktop\Nutt_KMUTT_folder\7\KEN-5895\FILE240403-085108F_lane_change_found.MP4
Video properties: 2560x1440 @ 30 FPS, Total frames: 1800
Starting video processing...
Processing frame 100/1800 (5.6%)
Processing frame 200/1800 (11.1%)
Processing frame 300/1800 (16.7%)
Processing frame 400/1800 (22.2%)
Processing frame 500/1800 (27.8%)
Processing frame 600/1800 (33.3%)
Processing frame 700/1800 (38.9%)
Processing frame 800/1800 (44.4%)
Processing frame 900/1800 (50.0%)
Processing frame 1000/1800 (55.6%)
Processing frame 1100/1800 (61.1%)
Processing frame 1200/1800 (66.7%)
Processing frame 1300/1800 (72.2%)
Processing frame 1400/1800 (77.8%)
Processing frame 1500/1800 (83.3%)
Processing frame 1600/1800 (88.9%)
Processing frame 1700/1800 (94.4%)
Processin

In [9]:
# Alternative Usage - Process Different Video Files
# You can try different videos from your workspace

# Example with different video files from your folders:
alternative_videos = [
    r"c:\Users\Taber\Desktop\Nutt_KMUTT_folder\2\737505456.564599.mp4",
    r"c:\Users\Taber\Desktop\Nutt_KMUTT_folder\3\739790513.565241.mp4", 
    r"c:\Users\Taber\Desktop\Nutt_KMUTT_folder\4\1130327--910-G6.MOV",
    r"c:\Users\Taber\Desktop\Nutt_KMUTT_folder\5\737795575.325078.mp4"
]

# Uncomment and modify one of these to process a different video:
# input_video = alternative_videos[0]  # Change index 0 to 1, 2, 3 for different videos
# output_video = "vehicle_tracking_output_2.mp4"

# Or manually set your video path:
# input_video = r"path\to\your\video\file.mp4"
# output_video = "your_output_name.mp4"

print("Alternative video options available above.")
print("Uncomment and modify the lines above to process different videos.")

Alternative video options available above.
Uncomment and modify the lines above to process different videos.


In [9]:
# Customization and Troubleshooting

# If you have your own trained model, update the model_path:
# model_path = r"path\to\your\custom\model.pt"

# Adjust tracking parameters if needed:
def create_custom_tracker(max_disappeared=30, max_distance=150):
    """
    Create tracker with custom parameters
    
    Args:
        max_disappeared (int): Frames before removing lost tracks (higher = more persistent tracking)
        max_distance (float): Max distance for track association (higher = more flexible matching)
    """
    return HomemadeStrongSORT(max_disappeared=max_disappeared, max_distance=max_distance)

# Troubleshooting Tips:
print("🔧 Troubleshooting Guide:")
print("1. If too many false detections: Increase min_thresh (try 0.4 or 0.5)")
print("2. If missing vehicles: Decrease min_thresh (try 0.2 or 0.25)")
print("3. If tracking is jumpy: Increase max_distance in tracker (try 200)")
print("4. If tracks persist too long: Decrease max_disappeared (try 15-20)")
print("5. If using custom model: Update vehicle_classes dictionary with your class IDs")

# Performance Tips:
print("\\n⚡ Performance Tips:")
print("1. For faster processing: Use smaller YOLO model (yolo11n.pt vs yolo11s.pt)")
print("2. For better accuracy: Use larger YOLO model (yolo11m.pt or yolo11l.pt)")
print("3. Adjust input video resolution if needed")
print("4. Consider processing every Nth frame for faster results (modify frame skip)")

# Custom vehicle classes (if using your own model):
# custom_vehicle_classes = {
#     0: 'car',
#     1: 'truck', 
#     2: 'bus',
#     3: 'motorcycle'
# }
# 
# custom_display_names = {
#     0: 'Car',
#     1: 'Truck',
#     2: 'Bus', 
#     3: 'Motorcycle'
# }

print("\\nCustomization options and troubleshooting tips displayed above.")

🔧 Troubleshooting Guide:
1. If too many false detections: Increase min_thresh (try 0.4 or 0.5)
2. If missing vehicles: Decrease min_thresh (try 0.2 or 0.25)
3. If tracking is jumpy: Increase max_distance in tracker (try 200)
4. If tracks persist too long: Decrease max_disappeared (try 15-20)
5. If using custom model: Update vehicle_classes dictionary with your class IDs
\n⚡ Performance Tips:
1. For faster processing: Use smaller YOLO model (yolo11n.pt vs yolo11s.pt)
2. For better accuracy: Use larger YOLO model (yolo11m.pt or yolo11l.pt)
3. Adjust input video resolution if needed
4. Consider processing every Nth frame for faster results (modify frame skip)
\nCustomization options and troubleshooting tips displayed above.


## Additional Features:Lane detector and vehicle counting

## environment Usage
- amazingclrer_dem (python 3.11.13)
- Based on the mmdetection library for lane detection and vehicle counting.
## notification
- this script is designed to work with the YOLOv11 and it was mytical pull project

In [13]:
# Manual extraction from your specific notebook code
required_packages = {
    # Core Python packages (usually pre-installed)
    'os': 'Built-in',
    'sys': 'Built-in', 
    'collections': 'Built-in',
    'math': 'Built-in',
    'logging': 'Built-in',
    'tempfile': 'Built-in',
    're': 'Built-in',
    'json': 'Built-in',
    
    # External packages that need installation
    'cv2': 'pip install opencv-python',
    'numpy': 'pip install numpy', 
    'torch': 'pip install torch',
    'torchvision': 'pip install torchvision',
    'ultralytics': 'pip install ultralytics',
    
    # CLRerNet dependencies (from your CLRerNet setup)
    'mmdet': 'pip install mmdet (part of CLRerNet setup)',
    'mmcv': 'pip install mmcv-full (part of CLRerNet setup)',
    
    # Additional dependencies
    'PIL': 'pip install Pillow',
    'matplotlib': 'pip install matplotlib (if needed for visualization)',
}

print("📋 Required packages for your notebook:")
print("\n🏠 Built-in Python modules:")
for pkg, install_cmd in required_packages.items():
    if 'Built-in' in install_cmd:
        print(f"  ✅ {pkg} - {install_cmd}")

print("\n📦 External packages to install:")
for pkg, install_cmd in required_packages.items():
    if 'pip install' in install_cmd:
        print(f"  📥 {pkg} - {install_cmd}")

📋 Required packages for your notebook:

🏠 Built-in Python modules:
  ✅ os - Built-in
  ✅ sys - Built-in
  ✅ collections - Built-in
  ✅ math - Built-in
  ✅ logging - Built-in
  ✅ tempfile - Built-in
  ✅ re - Built-in
  ✅ json - Built-in

📦 External packages to install:
  📥 cv2 - pip install opencv-python
  📥 numpy - pip install numpy
  📥 torch - pip install torch
  📥 torchvision - pip install torchvision
  📥 ultralytics - pip install ultralytics
  📥 mmdet - pip install mmdet (part of CLRerNet setup)
  📥 mmcv - pip install mmcv-full (part of CLRerNet setup)
  📥 PIL - pip install Pillow
  📥 matplotlib - pip install matplotlib (if needed for visualization)


In [14]:
# CLRerNet Lane Detection Integration

import sys
import tempfile
import os
from collections import defaultdict, deque

# Add CLRerNet to Python path
clrernet_path = r'C:\Users\Taber\Desktop\Nutt_KMUTT_folder\CLRerNet'
if clrernet_path not in sys.path:
    sys.path.insert(0, clrernet_path)

# CLRerNet Configuration
CLRERNET_CONFIG = r'C:\Users\Taber\Desktop\Nutt_KMUTT_folder\CLRerNet\configs\clrernet\culane\clrernet_culane_dla34.py'
CLRERNET_CHECKPOINT = r'C:\Users\Taber\Desktop\Nutt_KMUTT_folder\CLRerNet\clrernet_culane_dla34_ema.pth'
TARGET_WIDTH = 1640
TARGET_HEIGHT = 590

# Check if CLRerNet files exist
config_exists = os.path.exists(CLRERNET_CONFIG)
checkpoint_exists = os.path.exists(CLRERNET_CHECKPOINT)

print(f"CLRerNet config exists: {config_exists}")
print(f"CLRerNet checkpoint exists: {checkpoint_exists}")

if not config_exists:
    print(f"Config not found at: {CLRERNET_CONFIG}")
    # Alternative configs you might have
    alternative_configs = """
    Alternative config paths to try:
    - C:/Users/Taber/Desktop/Nutt_KMUTT_folder/CLRerNet/configs/clrernet/clrernet_r18_culane.py
    - C:/Users/Taber/Desktop/Nutt_KMUTT_folder/CLRerNet/configs/clrernet/clrernet_r34_culane.py
    """
    print(alternative_configs)

if not checkpoint_exists:
    print(f"Checkpoint not found at: {CLRERNET_CHECKPOINT}")
    print("Please make sure you have downloaded the CLRerNet checkpoint file")

# Try to import CLRerNet components with fallback
clrernet_available = False
try:
    from mmdet.apis import init_detector
    from libs.api.inference import inference_one_image
    from libs.utils.visualiser import visualize_lanes
    clrernet_available = True
    print("✅ CLRerNet components imported successfully!")
except ImportError as e:
    print(f"❌ CLRerNet import error: {e}")
    print("⚠️ CLRerNet not available - will use basic lane detection fallback")
    

CLRerNet config exists: True
CLRerNet checkpoint exists: True
✅ CLRerNet components imported successfully!


In [15]:
# Enhanced Lane Change Detection System with Bottom Line Method
class LaneChangeDetector:
    def __init__(self, lane_change_threshold=18):
        """
        Initialize lane change detector with bottom line method
        
        Args:
            lane_change_threshold (int): Minimum distance in pixels for lane change detection
        """
        self.lane_change_threshold = lane_change_threshold
        self.vehicle_lane_history = {}  # Track which lane each vehicle is in
        self.lane_changes = {}  # Track detected lane changes
        self.frame_count = 0
        
    def prepare_frame_for_lane_detection(self, frame):
        """Prepare frame for CLRerNet lane detection"""
        if frame is None:
            return None
        
        h, w, _ = frame.shape
        target_size = (TARGET_WIDTH, TARGET_HEIGHT)
        
        if w != TARGET_WIDTH or h != TARGET_HEIGHT:
            prepared_frame = cv2.resize(frame, target_size, interpolation=cv2.INTER_AREA)
        else:
            prepared_frame = frame
            
        return prepared_frame
    
    def detect_lanes_clrernet(self, frame, model):
        """
        Detect lanes using CLRerNet
        
        Args:
            frame (np.ndarray): Input frame
            model: CLRerNet model
            
        Returns:
            list: Detected lane lines as list of points
        """
        try:
            # Prepare frame
            prepared_frame = self.prepare_frame_for_lane_detection(frame)
            if prepared_frame is None:
                return []
            
            temp_image_path = None
            try:
                # Save frame to temporary file for CLRerNet processing
                with tempfile.NamedTemporaryFile(suffix='.png', delete=False) as temp_f:
                    temp_image_path = temp_f.name
                    cv2.imwrite(temp_image_path, prepared_frame)
                
                # Perform lane detection
                src, preds = inference_one_image(model, temp_image_path)
                
                # Scale predictions back to original frame size
                original_h, original_w = frame.shape[:2]
                scaled_preds = []
                
                for lane in preds:
                    scaled_lane = []
                    for point in lane:
                        x, y = point
                        # Scale from CLRerNet input size back to original frame size
                        scaled_x = int(x * original_w / TARGET_WIDTH)
                        scaled_y = int(y * original_h / TARGET_HEIGHT)
                        scaled_lane.append((scaled_x, scaled_y))
                    if len(scaled_lane) > 0:
                        scaled_preds.append(scaled_lane)
                
                return scaled_preds
                
            finally:
                # Clean up temporary file
                if temp_image_path and os.path.exists(temp_image_path):
                    os.remove(temp_image_path)
                    
        except Exception as e:
            print(f"Lane detection error: {e}")
            return []
    
    def get_vehicle_bottom_line(self, bbox):
        """
        Calculate the bottom line of vehicle with half-width on each side of center
        
        Args:
            bbox (tuple): Vehicle bounding box (x1, y1, x2, y2)
            
        Returns:
            tuple: (start_point, end_point) of the bottom line
        """
        x1, y1, x2, y2 = bbox
        
        # Calculate center bottom point
        center_x = int((x1 + x2) / 2)
        bottom_y = int(y2)
        
        # Calculate half of the bbox width
        bbox_width = x2 - x1
        half_width = int(bbox_width / 4)  # Using 1/4 on each side (total 1/2 width)
        
        # Create line endpoints
        line_start = (center_x - half_width, bottom_y)
        line_end = (center_x + half_width, bottom_y)
        
        return line_start, line_end
    
    def line_intersects_lane(self, line_start, line_end, lane_points):
        """
        Check if the vehicle bottom line intersects with any lane line segment
        
        Args:
            line_start (tuple): Start point of vehicle bottom line
            line_end (tuple): End point of vehicle bottom line
            lane_points (list): List of lane line points
            
        Returns:
            bool: True if intersection exists, False otherwise
        """
        if len(lane_points) < 2:
            return False
        
        # Vehicle bottom line (horizontal)
        veh_x1, veh_y = line_start
        veh_x2, _ = line_end
        
        # Check intersection with each lane segment
        for i in range(len(lane_points) - 1):
            lane_x1, lane_y1 = lane_points[i]
            lane_x2, lane_y2 = lane_points[i + 1]
            
            # Check if the horizontal vehicle line intersects with lane segment
            # Vehicle line is at y = veh_y from x = veh_x1 to x = veh_x2
            
            # Find intersection point
            if lane_y1 == lane_y2:  # Lane segment is horizontal
                if abs(lane_y1 - veh_y) < 5:  # Close enough to same y level
                    # Check if there's x overlap
                    lane_x_min, lane_x_max = min(lane_x1, lane_x2), max(lane_x1, lane_x2)
                    veh_x_min, veh_x_max = min(veh_x1, veh_x2), max(veh_x1, veh_x2)
                    
                    if not (lane_x_max < veh_x_min or lane_x_min > veh_x_max):
                        return True
            else:  # Lane segment is not horizontal
                # Calculate intersection of vehicle horizontal line with lane segment
                if lane_y1 != lane_y2:  # Avoid division by zero
                    # Parametric intersection calculation
                    t = (veh_y - lane_y1) / (lane_y2 - lane_y1)
                    
                    if 0 <= t <= 1:  # Intersection point is on the lane segment
                        intersect_x = lane_x1 + t * (lane_x2 - lane_x1)
                        
                        # Check if intersection x is within vehicle line range
                        if min(veh_x1, veh_x2) <= intersect_x <= max(veh_x1, veh_x2):
                            return True
        
        return False
    
    def find_lanes_crossed_by_vehicle(self, vehicle_bottom_line, lanes):
        """
        Find which lanes are crossed by the vehicle's bottom line
        
        Args:
            vehicle_bottom_line (tuple): (start_point, end_point) of vehicle bottom line
            lanes (list): List of lane lines
            
        Returns:
            list: Indices of lanes that are crossed by the vehicle line
        """
        crossed_lanes = []
        line_start, line_end = vehicle_bottom_line
        
        for lane_idx, lane_points in enumerate(lanes):
            if self.line_intersects_lane(line_start, line_end, lane_points):
                crossed_lanes.append(lane_idx)
        
        return crossed_lanes
    
    def detect_lane_change(self, vehicle_id, crossed_lanes):
        """
        Detect if a vehicle has changed lanes based on crossed lane lines
        
        Args:
            vehicle_id (int): Unique vehicle ID
            crossed_lanes (list): List of lane indices that vehicle line crosses
            
        Returns:
            bool: True if lane change detected, False otherwise
        """
        if vehicle_id not in self.vehicle_lane_history:
            # First time seeing this vehicle
            self.vehicle_lane_history[vehicle_id] = deque(maxlen=10)
            self.vehicle_lane_history[vehicle_id].append(tuple(sorted(crossed_lanes)))
            return False
        
        # Add current crossed lanes to history
        current_crossed = tuple(sorted(crossed_lanes))
        self.vehicle_lane_history[vehicle_id].append(current_crossed)
        
        # Check for lane change - look for consistent change over several frames
        history = list(self.vehicle_lane_history[vehicle_id])
        if len(history) < 5:  # Need at least 5 frames of history
            return False
        
        # Compare old and new crossed lanes
        old_crossed = history[0]
        new_crossed = current_crossed
        
        # Lane change detected if:
        # 1. Vehicle is crossing different lanes now vs before
        # 2. There's actual movement between lane regions
        if old_crossed != new_crossed and len(old_crossed) > 0 and len(new_crossed) > 0:
            # Check if the change is consistent (not just noise)
            recent_crossed = history[-3:]  # Last 3 frames
            if all(crossed == new_crossed for crossed in recent_crossed):
                # Lane change detected!
                if vehicle_id not in self.lane_changes:
                    self.lane_changes[vehicle_id] = []
                
                self.lane_changes[vehicle_id].append({
                    'frame': self.frame_count,
                    'from_lanes': old_crossed,
                    'to_lanes': new_crossed
                })
                
                # Reset history for this vehicle
                self.vehicle_lane_history[vehicle_id] = deque(maxlen=10)
                self.vehicle_lane_history[vehicle_id].append(new_crossed)
                
                return True
        
        return False
    
    def draw_lanes_and_detection(self, frame, lanes, tracked_objects, lane_changes_detected):
        """
        Draw lanes and lane change detection information on frame
        
        Args:
            frame (np.ndarray): Input frame
            lanes (list): Detected lane lines
            tracked_objects (dict): Currently tracked vehicles
            lane_changes_detected (dict): Vehicles with detected lane changes
            
        Returns:
            np.ndarray: Frame with overlays
        """
        # Draw detected lanes (green lines)
        for lane_idx, lane in enumerate(lanes):
            if len(lane) > 1:
                points = np.array(lane, dtype=np.int32)
                cv2.polylines(frame, [points], False, (0, 255, 0), 3)  # Green lanes
                
                # Label lane number
                if len(lane) > 0:
                    start_point = lane[0]
                    cv2.putText(frame, f'L{lane_idx}', 
                              (start_point[0], start_point[1] - 10),
                              cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
        
        # Draw vehicle bottom lines and lane crossing detection
        for object_id, obj_data in tracked_objects.items():
            x1, y1, x2, y2 = obj_data['bbox']
            
            # Get vehicle bottom line
            line_start, line_end = self.get_vehicle_bottom_line((x1, y1, x2, y2))
            
            # Find which lanes this vehicle line crosses
            crossed_lanes = self.find_lanes_crossed_by_vehicle((line_start, line_end), lanes)
            
            # Draw bottom line
            line_color = (255, 0, 255)  # Magenta for normal
            line_thickness = 3
            
            # Special highlighting for lane changes
            if object_id in lane_changes_detected:
                line_color = (0, 0, 255)  # Red for lane change
                line_thickness = 5
            
            # Draw the bottom line
            cv2.line(frame, line_start, line_end, line_color, line_thickness)
            
            # Draw endpoints of the line
            cv2.circle(frame, line_start, 4, line_color, -1)
            cv2.circle(frame, line_end, 4, line_color, -1)
            
            # Check if this vehicle has a lane change detected
            if object_id in lane_changes_detected:
                # Draw special indicator for lane change
                cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), (0, 0, 255), 4)  # Red box
                cv2.putText(frame, 'LANE CHANGE!', 
                          (int(x1), int(y1) - 30),
                          cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 3)
            
            # Show crossed lanes info
            if crossed_lanes:
                info_text = f'Crossing: {crossed_lanes}'
                cv2.putText(frame, info_text, 
                          (int(x1), int(y2) + 20),
                          cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
        
        # Draw lane change information panel
        self.draw_lane_change_panel(frame, lane_changes_detected)
        
        return frame
    
    def draw_lane_change_panel(self, frame, lane_changes_detected):
        """Draw information panel showing lane change detections"""
        panel_height = 100
        panel_width = 500
        
        # Draw semi-transparent background
        overlay = frame.copy()
        cv2.rectangle(overlay, (frame.shape[1] - panel_width - 10, 10), 
                     (frame.shape[1] - 10, panel_height), (0, 0, 0), -1)
        cv2.addWeighted(overlay, 0.7, frame, 0.3, 0, frame)
        
        # Draw border
        cv2.rectangle(frame, (frame.shape[1] - panel_width - 10, 10), 
                     (frame.shape[1] - 10, panel_height), (255, 255, 255), 2)
        
        y_offset = 35
        cv2.putText(frame, f'Lane Changes Detected: {len(lane_changes_detected)}', 
                    (frame.shape[1] - panel_width, y_offset), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
        
        y_offset += 25
        if lane_changes_detected:
            for vehicle_id in list(lane_changes_detected.keys())[:2]:  # Show last 2
                cv2.putText(frame, f'Vehicle {vehicle_id}: Line Crossed Lane', 
                           (frame.shape[1] - panel_width, y_offset), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
                y_offset += 20

print("✅ Enhanced Lane Change Detection with Bottom Line Method implemented!")
print("Features:")
print("  • Bottom line spans half the vehicle width")
print("  • Detects when entire line crosses lane boundaries") 
print("  • More accurate lane change detection")
print("  • Visual feedback with colored lines")
print("  • Robust intersection calculation")

✅ Enhanced Lane Change Detection with Bottom Line Method implemented!
Features:
  • Bottom line spans half the vehicle width
  • Detects when entire line crosses lane boundaries
  • More accurate lane change detection
  • Visual feedback with colored lines
  • Robust intersection calculation


In [19]:
#additional_frame_preparation
#Function to prepare clahe enhancement incase of low light conditions
def prepare_frame_for_clahe(frame):
    """
    Prepare frame for CLAHE enhancement
    
    Args:
        frame (np.ndarray): Input frame
        
    Returns:
        np.ndarray: Enhanced frame
    """
    if frame is None:
        return None
    
    # Convert to YUV color space
    yuv_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2YUV)
    
    # Apply CLAHE to the Y channel
    y_channel = yuv_frame[:, :, 0]
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced_y_channel = clahe.apply(y_channel)
    
    # Merge back with U and V channels
    yuv_frame[:, :, 0] = enhanced_y_channel
    enhanced_frame = cv2.cvtColor(yuv_frame, cv2.COLOR_YUV2BGR)
    
    return enhanced_frame


# Enhanced Vehicle Tracking with Bottom Line Lane Change Detection
def process_video_with_lane_change_detection(input_path, output_path, model_path, 
                                           clrernet_config, clrernet_checkpoint, 
                                           min_confidence, device='cuda:0'):
    """
    Process video with vehicle detection, tracking, and bottom line lane change detection
    """
    
    # Check if all required files exist
    required_files = [
        (model_path, "YOLO model"),
        (input_path, "Input video"),
        (clrernet_config, "CLRerNet config"),
        (clrernet_checkpoint, "CLRerNet checkpoint")
    ]
    
    for file_path, file_type in required_files:
        if not os.path.exists(file_path):
            print(f'ERROR: {file_type} not found at {file_path}')
            return False
    
    print("Loading models...")
    
    # Load YOLO model
    yolo_model = YOLO(model_path, task='detect')
    print("✅ YOLO model loaded")
    
    # Load CLRerNet model
    try:
        clrernet_model = init_detector(clrernet_config, clrernet_checkpoint, device=device)
        print("✅ CLRerNet model loaded")
    except Exception as e:
        print(f"❌ Failed to load CLRerNet model: {e}")
        return False
    
    # Initialize video capture
    cap = cv2.VideoCapture(input_path)
    if not cap.isOpened():
        print("ERROR: Could not open input video file.")
        return False
    
    # Get video properties
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    print(f"Video properties: {width}x{height} @ {fps} FPS, Total frames: {total_frames}")
    
    # Initialize video writer
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))
    
    # Initialize tracker and lane change detector
    tracker = HomemadeStrongSORT(max_disappeared=30, max_distance=150)
    lane_detector = LaneChangeDetector(lane_change_threshold=18)
    
    # Initialize counters
    frame_count = 0
    total_vehicles_ever_seen = set()
    vehicle_counts = defaultdict(int)
    bbox_colors = get_bbox_colors()
    lane_changes_in_current_frame = {}
    
    print("Starting video processing with bottom line lane change detection...")
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break
            
        frame_count += 1
        lane_detector.frame_count = frame_count
        
        # Print progress every 100 frames
        if frame_count % 100 == 0:
            progress = (frame_count / total_frames) * 100
            print(f"Processing frame {frame_count}/{total_frames} ({progress:.1f}%)")
        # Step 0.5 = Prepare frame for CLAHE enhancement if needed
        frame = prepare_frame_for_clahe(frame) # Optional step for low light conditions
        # Step 1: Detect lanes using CLRerNet
        detected_lanes = lane_detector.detect_lanes_clrernet(frame, clrernet_model)
        
        # Step 2: Run YOLO vehicle detection
        results = yolo_model(frame, verbose=False)
        detections = results[0].boxes
        
        # Process vehicle detections
        current_detections = []
        if detections is not None and len(detections) > 0:
            for i in range(len(detections)):
                xyxy_tensor = detections[i].xyxy.cpu()
                xyxy = xyxy_tensor.numpy().squeeze()
                x1, y1, x2, y2 = xyxy.astype(int)
                
                class_id = int(detections[i].cls.item())
                confidence = detections[i].conf.item()
                
                # Only process vehicle classes with sufficient confidence
                if class_id in vehicle_classes and confidence > min_confidence:
                    current_detections.append([x1, y1, x2, y2, confidence, class_id])
        
        # Step 3: Update tracker
        tracked_objects = tracker.update(current_detections)
        
        # Step 4: Perform lane change detection using bottom line method
        lane_changes_in_current_frame = {}
        
        for object_id, obj_data in tracked_objects.items():
            total_vehicles_ever_seen.add(object_id)
            
            x1, y1, x2, y2 = obj_data['bbox']
            class_id = obj_data['class_id']
            
            # Get vehicle bottom line
            vehicle_bottom_line = lane_detector.get_vehicle_bottom_line((x1, y1, x2, y2))
            
            # Find which lanes this vehicle's bottom line crosses
            crossed_lanes = lane_detector.find_lanes_crossed_by_vehicle(vehicle_bottom_line, detected_lanes)
            
            # Check for lane change
            lane_change_detected = lane_detector.detect_lane_change(object_id, crossed_lanes)
            
            if lane_change_detected:
                lane_changes_in_current_frame[object_id] = True
                print(f"🚨 LANE CHANGE DETECTED: Vehicle {object_id} at frame {frame_count}")
                print(f"   Vehicle bottom line crossed lanes: {crossed_lanes}")
        
        # Step 5: Draw everything on the frame
        
        # Draw vehicle tracking (existing functionality)
        current_frame_counts = defaultdict(int)
        for object_id, obj_data in tracked_objects.items():
            x1, y1, x2, y2 = obj_data['bbox']
            class_id = obj_data['class_id']
            centroid = obj_data['centroid']
            trail = obj_data['trail']
            confidence = obj_data.get('confidence', 0)
            
            current_frame_counts[class_id] += 1
            color = bbox_colors[object_id % len(bbox_colors)]
            
            # Special highlighting for lane changes
            if object_id in lane_changes_in_current_frame:
                color = (0, 0, 255)  # Red for lane change
                thickness = 4
            else:
                thickness = 2
            
            # Draw bounding box
            cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), color, thickness)
            
            # Draw tracking trail
            if len(trail) > 1:
                draw_trail(frame, trail, color, thickness=2)
            
            # Draw vehicle ID and class label
            if class_id in vehicle_display_names:
                label = f'ID:{object_id} {vehicle_display_names[class_id]}'
                if object_id in lane_changes_in_current_frame:
                    label += ' [LINE CROSSED LANE!]'
                
                label_size, _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
                
                # Draw label background
                cv2.rectangle(frame, (int(x1), int(y1) - label_size[1] - 10), 
                             (int(x1) + label_size[0], int(y1)), color, -1)
                
                # Draw label text
                cv2.putText(frame, label, (int(x1), int(y1) - 5), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
                
                # Draw center point
                cv2.circle(frame, centroid, 4, color, -1)
        
        # Draw lanes and bottom line lane change detection visualization
        frame = lane_detector.draw_lanes_and_detection(
            frame, detected_lanes, tracked_objects, lane_changes_in_current_frame
        )
        
        # Update vehicle counts
        for class_id, count in current_frame_counts.items():
            vehicle_counts[class_id] = max(vehicle_counts[class_id], count)
        
        # Write frame to output video
        out.write(frame)
    
    # Clean up
    cap.release()
    out.release()
    
    # Print final statistics
    print(f"\n🎯 Video processing completed!")
    print(f"📁 Output saved to: {output_path}")
    print(f"🚗 Total unique vehicles tracked: {len(total_vehicles_ever_seen)}")
    print(f"🔄 Total lane changes detected: {sum(len(changes) for changes in lane_detector.lane_changes.values())}")
    
    print("\n📊 Vehicle type breakdown:")
    for class_id, count in vehicle_counts.items():
        if class_id in vehicle_display_names:
            print(f"  {vehicle_display_names[class_id]}: {count}")
    
    print("\n🔄 Lane change summary:")
    if lane_detector.lane_changes:
        for vehicle_id, changes in lane_detector.lane_changes.items():
            print(f"  Vehicle {vehicle_id}: {len(changes)} lane changes")
            for change in changes:
                print(f"    Frame {change['frame']}: Lanes {change['from_lanes']} → {change['to_lanes']}")
    else:
        print("  No lane changes detected")
    
    return True

print("✅ Enhanced processing function with bottom line lane change detection ready!")
print("🎯 New features:")
print("  • Bottom horizontal line spans half vehicle width")
print("  • Detects when entire line crosses lane boundaries")
print("  • More accurate than single point method")
print("  • Visual feedback with colored lines and intersection info")
print("  • Robust geometric intersection calculations")

✅ Enhanced processing function with bottom line lane change detection ready!
🎯 New features:
  • Bottom horizontal line spans half vehicle width
  • Detects when entire line crosses lane boundaries
  • More accurate than single point method
  • Visual feedback with colored lines and intersection info
  • Robust geometric intersection calculations


In [20]:
# Execute Lane Change Detection on Video
# Set your input video path here
test_input_video = r"C:\Users\Taber\Desktop\Nutt_KMUTT_folder\5\737795575.325078.mp4"
test_output_video = r"C:\Users\Taber\Desktop\Nutt_KMUTT_folder\5\737795575.325078_high_noon_clahr.mp4"
model_path = r"C:\Users\Taber\Desktop\Nutt_KMUTT_folder\Vehicle_tracking\yolo11x.pt"
min_thresh = 0.5  # Minimum confidence threshold for detections
import torch
from ultralytics import YOLO
import cv2
import math
import numpy as np
# Check if all required files exist
print("🔍 Checking required files...")
print(f"Input video: {os.path.exists(test_input_video)} - {test_input_video}")
print(f"YOLO model: {os.path.exists(model_path)} - {model_path}")
print(f"CLRerNet config: {os.path.exists(CLRERNET_CONFIG)} - {CLRERNET_CONFIG}")
print(f"CLRerNet checkpoint: {os.path.exists(CLRERNET_CHECKPOINT)} - {CLRERNET_CHECKPOINT}")

# Process the video if all files exist
if (os.path.exists(test_input_video) and 
    os.path.exists(model_path) and 
    os.path.exists(CLRERNET_CONFIG) and 
    os.path.exists(CLRERNET_CHECKPOINT)):
    
    print("\\n🚀 Starting lane change detection processing...")
    
    success = process_video_with_lane_change_detection(
        input_path=test_input_video,
        output_path=test_output_video,
        model_path=model_path,
        clrernet_config=CLRERNET_CONFIG,
        clrernet_checkpoint=CLRERNET_CHECKPOINT,
        min_confidence=min_thresh,
        device='cuda:0' if torch.cuda.is_available() else 'cpu'
    )
    
    if success:
        print("\\n✅ Lane change detection completed successfully!")
        print(f"📁 Output video: {test_output_video}")
        print("\\n🎯 Features included in output:")
        print("   • Vehicle detection and tracking with unique IDs")
        print("   • Robust lane detection (green lines)")
        print("   • Vehicle bottom-center tracking (magenta dots)")
        print("   • Lane change detection with red highlighting")
        print("   • Real-time lane change alerts")
        print("   • Lane change statistics panel")
    else:
        print("❌ Lane change detection processing failed!")

else:
    print("\\n❌ Missing required files!")
    print("\\n📋 Required files for lane change detection:")
    print("1. Input video file")
    print("2. YOLO model (.pt file)")
    print("3. CLRerNet config file (.py)")
    print("4. CLRerNet checkpoint file (.pth)")
    
    print("\\n💡 To get CLRerNet files:")
    print("1. Download CLRerNet checkpoint from their GitHub repository")
    print("2. Ensure the config file path is correct")
    print("3. Update the CLRERNET_CONFIG and CLRERNET_CHECKPOINT paths above")
    
    # List available video files as alternatives
    print("\\n📁 Available video files in workspace:")
    video_extensions = ['.mp4', '.avi', '.mkv', '.mov', '.MOV', '.MP4']
    video_files = []
    
    for root, dirs, files in os.walk(r"c:\Users\Taber\Desktop\Nutt_KMUTT_folder"):
        for file in files:
            if any(file.lower().endswith(ext.lower()) for ext in video_extensions):
                video_files.append(os.path.join(root, file))
    
    for i, video_file in enumerate(video_files[:5]):  # Show first 5 videos
        print(f"   {i+1}. {video_file}")
    
    if len(video_files) > 5:
        print(f"   ... and {len(video_files) - 5} more videos")

🔍 Checking required files...
Input video: True - C:\Users\Taber\Desktop\Nutt_KMUTT_folder\5\737795575.325078.mp4
YOLO model: True - C:\Users\Taber\Desktop\Nutt_KMUTT_folder\Vehicle_tracking\yolo11x.pt
CLRerNet config: True - C:\Users\Taber\Desktop\Nutt_KMUTT_folder\CLRerNet\configs\clrernet\culane\clrernet_culane_dla34.py
CLRerNet checkpoint: True - C:\Users\Taber\Desktop\Nutt_KMUTT_folder\CLRerNet\clrernet_culane_dla34_ema.pth
\n🚀 Starting lane change detection processing...
Loading models...
✅ YOLO model loaded
Loads checkpoint by local backend from path: C:\Users\Taber\Desktop\Nutt_KMUTT_folder\CLRerNet\clrernet_culane_dla34_ema.pth
The model and loaded state dict do not match exactly

missing keys in source state_dict: bbox_head.loss_seg.criterion.weight, bbox_head.seg_decoder.conv.weight, bbox_head.seg_decoder.conv.bias

34680 data are loaded
✅ CLRerNet model loaded
Video properties: 1280x720 @ 30 FPS, Total frames: 1830
Starting video processing with bottom line lane change detec

c:\Users\Taber\miniconda3\envs\amazingclrer_dem\Lib\site-packages\mmdet\apis\inference.py:90: UserWarning: dataset_meta or class names are not saved in the checkpoint's meta data, use COCO classes by default.
  warnings.warn(
c:\Users\Taber\miniconda3\envs\amazingclrer_dem\Lib\site-packages\mmdet\apis\inference.py:108: UserWarning: palette does not exist, random is used by default. You can also set the palette to customize.
  warnings.warn(


Processing frame 100/1830 (5.5%)
Processing frame 200/1830 (10.9%)
Processing frame 300/1830 (16.4%)
Processing frame 400/1830 (21.9%)
Processing frame 500/1830 (27.3%)
Processing frame 600/1830 (32.8%)
Processing frame 700/1830 (38.3%)
Processing frame 800/1830 (43.7%)
Processing frame 900/1830 (49.2%)
Processing frame 1000/1830 (54.6%)
Processing frame 1100/1830 (60.1%)
Processing frame 1200/1830 (65.6%)
Processing frame 1300/1830 (71.0%)
Processing frame 1400/1830 (76.5%)
Processing frame 1500/1830 (82.0%)
Processing frame 1600/1830 (87.4%)
Processing frame 1700/1830 (92.9%)
Processing frame 1800/1830 (98.4%)

🎯 Video processing completed!
📁 Output saved to: C:\Users\Taber\Desktop\Nutt_KMUTT_folder\5\737795575.325078_high_noon_clahr.mp4
🚗 Total unique vehicles tracked: 14
🔄 Total lane changes detected: 0

📊 Vehicle type breakdown:
  Truck: 4
  Bus: 2
  Car: 2

🔄 Lane change summary:
  No lane changes detected
\n✅ Lane change detection completed successfully!
📁 Output video: C:\Users

In [23]:
# Lane Change Detection Customization and Tips

# Custom lane detector with different sensitivity
def create_custom_lane_detector(lane_change_threshold=15, history_frames=10 ):
    """
    Create lane change detector with custom parameters
    
    Args:
        lane_change_threshold (int): Distance threshold for lane assignment (pixels)
        history_frames (int): Number of frames to keep in history for tracking
    """
    detector = LaneChangeDetector(lane_change_threshold=lane_change_threshold)
    # You can modify internal parameters here if needed
    return detector

# Custom tracker with optimized parameters for lane change detection
def create_custom_tracker(max_disappeared=20, max_distance=100):
    """
    Create vehicle tracker optimized for lane change detection
    
    Args:
        max_disappeared (int): Frames before removing lost tracks
        max_distance (float): Max distance for track association
    """
    return HomemadeStrongSORT(max_disappeared=max_disappeared, max_distance=max_distance)

# Processing tips and customization options
processing_tips = """
🔧 Lane Change Detection Tuning Guide:

1. LANE DETECTION SENSITIVITY:
   - If lanes are not detected properly: Check CLRerNet model and config
   - If too many false lanes: The CLRerNet model is robust, shouldn't need changes
   - Lane detection uses the proven CLRerNet system (don't fix what works!)

2. LANE CHANGE SENSITIVITY:
   - Too many false lane changes: Increase lane_change_threshold (try 15-20)
   - Missing real lane changes: Decrease lane_change_threshold (try 5-8)
   - Noisy detection: Increase history_frames requirement (try 7-10)

3. VEHICLE TRACKING:
   - Lost vehicles: Increase max_disappeared (try 40-50)
   - Wrong associations: Decrease max_distance (try 80-120)
   - Jumpy tracking: Increase max_distance (try 150-200)

4. PERFORMANCE OPTIMIZATION:
   - Faster processing: Skip some frames for lane detection
   - Better accuracy: Use higher resolution or better YOLO model
   - Memory usage: Reduce history tracking or trail length

5. VISUALIZATION:
   - Lane colors: Modify draw_lanes_and_detection function
   - Alert styling: Customize lane change highlighting
   - Information panels: Adjust panel size and content
"""

print(processing_tips)

# Alternative processing for different scenarios
def process_highway_lane_changes(input_path, output_path):
    """Highway-optimized lane change detection"""
    return process_video_with_lane_change_detection(
        input_path=input_path,
        output_path=output_path,
        model_path=model_path,
        clrernet_config=CLRERNET_CONFIG,
        clrernet_checkpoint=CLRERNET_CHECKPOINT,
        min_confidence=0.35,  # Higher confidence for highway
        device='cuda:0' if torch.cuda.is_available() else 'cpu'
    )

def process_city_lane_changes(input_path, output_path):
    """City traffic optimized lane change detection"""
    return process_video_with_lane_change_detection(
        input_path=input_path,
        output_path=output_path,
        model_path=model_path,
        clrernet_config=CLRERNET_CONFIG,
        clrernet_checkpoint=CLRERNET_CHECKPOINT,
        min_confidence=0.25,  # Lower confidence for city traffic
        device='cuda:0' if torch.cuda.is_available() else 'cpu'
    )

# Troubleshooting common issues
troubleshooting_tips = """
🚨 Common Issues and Solutions:

1. "CLRerNet import error":
   - Ensure CLRerNet is properly installed
   - Check if mmdetection and mmcv are installed
   - Verify CUDA/PyTorch compatibility

2. "Lane detection not working":
   - Check CLRerNet checkpoint file exists and is not corrupted
   - Verify config file path is correct
   - Test with original CLRerNet demo first

3. "No lane changes detected":
   - Check if lanes are being detected (green lines should appear)
   - Verify vehicles are being tracked (should see bounding boxes)
   - Try lowering lane_change_threshold

4. "Too many false lane changes":
   - Increase detection threshold
   - Ensure proper video quality and lighting
   - Check if camera angle is suitable for lane detection

5. "Performance issues":
   - Use GPU if available (CUDA)
   - Reduce video resolution for processing
   - Consider processing every nth frame instead of all frames
"""

print(troubleshooting_tips)

print("\\n✅ Customization and troubleshooting guide loaded!")
print("💡 Use the custom functions above to fine-tune detection for your specific use case")


🔧 Lane Change Detection Tuning Guide:

1. LANE DETECTION SENSITIVITY:
   - If lanes are not detected properly: Check CLRerNet model and config
   - If too many false lanes: The CLRerNet model is robust, shouldn't need changes
   - Lane detection uses the proven CLRerNet system (don't fix what works!)

2. LANE CHANGE SENSITIVITY:
   - Too many false lane changes: Increase lane_change_threshold (try 15-20)
   - Missing real lane changes: Decrease lane_change_threshold (try 5-8)
   - Noisy detection: Increase history_frames requirement (try 7-10)

3. VEHICLE TRACKING:
   - Lost vehicles: Increase max_disappeared (try 40-50)
   - Wrong associations: Decrease max_distance (try 80-120)
   - Jumpy tracking: Increase max_distance (try 150-200)

4. PERFORMANCE OPTIMIZATION:
   - Faster processing: Skip some frames for lane detection
   - Better accuracy: Use higher resolution or better YOLO model
   - Memory usage: Reduce history tracking or trail length

5. VISUALIZATION:
   - Lane colors: Mo